# Phase 2.4 — Pandas & Data Handling

This notebook applies core Pandas data-handling operations to the project's actual NSE company-master and market datasets.

Topics covered:
- Loading Parquet datasets
- Inspecting dimensions, columns, and data types
- Summary statistics and missing values
- Categorical-value inspection
- Column selection and row filtering
- Sorting
- Trading-date inspection
- Duplicate validation
- Groupby and aggregation
- Dataset merging
- Datetime operations
- Daily returns with `pct_change()`
- Rolling averages and rolling volatility

This notebook is for data-handling learning and validation. Final investment targets and production ML features are created in later phases.

In [ ]:
from pathlib import Path
import pandas as pd

# Project paths
project_root = Path("/content/stock-investment-ml")
processed_dir = project_root / "data" / "processed" / "nse"
market_dir = processed_dir / "market"

# Discover available datasets
company_files = sorted(processed_dir.glob("*.parquet"))
market_files = sorted(market_dir.glob("*.parquet"))

if not company_files:
    raise FileNotFoundError(f"No company master Parquet file found in {processed_dir}")

if not market_files:
    raise FileNotFoundError(f"No market Parquet files found in {market_dir}")

# Load datasets
company_path = company_files[0]
market_path = market_files[-1]

company_df = pd.read_parquet(company_path)
market_df = pd.read_parquet(market_path)

# Basic validation
if company_df.empty:
    raise ValueError("Company master dataset is empty.")

if market_df.empty:
    raise ValueError("Market dataset is empty.")

print("Datasets loaded successfully.")
print(f"Company master : {company_df.shape}")
print(f"Market data    : {market_df.shape}")
print(f"Company file   : {company_path.name}")
print(f"Market file    : {market_path.name}")

In [ ]:
print("=== COMPANY MASTER COLUMNS ===")
print(company_df.columns.tolist())

print("\n=== MARKET DATA COLUMNS ===")
print(market_df.columns.tolist())

print("\n=== MARKET DATA TYPES ===")
print(market_df.dtypes)

In [ ]:
print("=== DATASET DIMENSIONS ===")
print(f"Company master: {company_df.shape[0]:,} rows × {company_df.shape[1]} columns")
print(f"Market data   : {market_df.shape[0]:,} rows × {market_df.shape[1]} columns")
        print("\n=== MARKET DATA SUMMARY STATISTICS ===")
display(market_df.describe())

print("\n=== MISSING VALUES ===")
display(market_df.isna().sum().to_frame("missing_count"))

In [ ]:
# Check categorical fields
categorical_cols = ["series", "instrument_type", "segment"]

for col in categorical_cols:
    counts = market_df[col].value_counts(dropna=False)
    print(f"\n=== {col.upper()} ===")
    print(f"Unique values: {counts.size}")
    print(counts)

In [ ]:
# Inspect trading-date structure
date_counts = market_df["trade_date"].value_counts().sort_index()

print("=== TRADING DATE SUMMARY ===")
print(f"Number of trading days: {date_counts.size}")
print(f"Date range: {date_counts.index.min().date()} to {date_counts.index.max().date()}")
print("\nRecords per trading day:")
print(date_counts)

In [ ]:
# Select specific columns
price_data = market_df[["nse_symbol", "company_name", "trade_date", "close"]]

print("=== SELECTED COLUMNS ===")
display(price_data.head())

# Filter rows using the latest available trading date
latest_date = market_df["trade_date"].max()
latest_data = market_df[market_df["trade_date"] == latest_date]

print(f"\n=== DATA FOR LATEST TRADE DATE: {latest_date.date()} ===")
print(f"Rows: {len(latest_data):,}")
display(latest_data[["nse_symbol", "company_name", "close"]].head())

In [ ]:
# Sort market data by closing price
sorted_market = market_df.sort_values("close", ascending=False)

print("=== HIGHEST CLOSING PRICES ===")
display(
    sorted_market[
        ["nse_symbol", "company_name", "trade_date", "close"]
    ].head(10)
)

print("\n=== LOWEST CLOSING PRICES ===")
display(
    market_df.sort_values("close", ascending=True)[
        ["nse_symbol", "company_name", "trade_date", "close"]
    ].head(10)
)

In [ ]:
# Data quality and groupby summaries

# Duplicate check
duplicate_count = market_df.duplicated(
    subset=["trade_date", "instrument_id"]
).sum()

print("=== DATA QUALITY CHECK ===")
print(f"Duplicate rows: {duplicate_count:,}")
print("No duplicates found." if duplicate_count == 0 else "Duplicates detected.")

# Missing-value check
missing = market_df.isna().sum()
missing = missing[missing > 0]

print("\n=== MISSING VALUES ===")
print("No missing values found." if missing.empty else missing)

# Daily market summary
daily_summary = (
    market_df
    .groupby("trade_date")
    .agg(
        companies=("instrument_id", "nunique"),
        total_volume=("volume", "sum"),
        total_turnover=("turnover", "sum"),
        total_trades=("number_of_trades", "sum")
    )
)

print("\n=== DAILY MARKET SUMMARY ===")
display(daily_summary)

# Company-level market summary
company_summary = (
    market_df
    .groupby(["nse_symbol", "company_name"])
    .agg(
        trading_days=("trade_date", "nunique"),
        average_close=("close", "mean"),
        average_volume=("volume", "mean"),
        average_turnover=("turnover", "mean")
    )
    .sort_values("average_turnover", ascending=False)
)

print("=== COMPANY-LEVEL MARKET SUMMARY ===")
display(company_summary.head(10))

In [ ]:
# Merge market data with company master using ISIN
merged_df = market_df.merge(
    company_df[["isin", "paid_up_value", "market_lot", "face_value"]],
    on="isin",
    how="left",
    validate="many_to_one"
)

print("=== MERGED DATASET ===")
print(f"Market data rows : {len(market_df):,}")
print(f"Merged rows      : {len(merged_df):,}")
print(f"Merged columns   : {len(merged_df.columns)}")

print("\n=== NEW COMPANY-MASTER FIELDS ===")
display(
    merged_df[
        [
            "nse_symbol",
            "company_name",
            "trade_date",
            "close",
            "paid_up_value",
            "market_lot",
            "face_value"
        ]
    ].head()
)

In [ ]:
# Verify that the merge matched every market record
new_fields = ["paid_up_value", "market_lot", "face_value"]
unmatched = merged_df[new_fields].isna().any(axis=1).sum()

print("=== MERGE VALIDATION ===")
print(f"Unmatched market rows: {unmatched:,}")

if unmatched == 0:
    print("All market records matched the company master.")
else:
    print("Some market records did not match the company master.")

In [ ]:
# Datetime feature extraction and range check
        date_features = merged_df[["trade_date"]].copy()

date_features["year"] = date_features["trade_date"].dt.year
date_features["month"] = date_features["trade_date"].dt.month
date_features["day"] = date_features["trade_date"].dt.day
date_features["weekday"] = date_features["trade_date"].dt.day_name()

print("=== DATE FEATURES ===")
display(date_features.head())

print("\n=== DATE RANGE ===")
print(f"Start: {merged_df['trade_date'].min().date()}")
print(f"End  : {merged_df['trade_date'].max().date()}")

In [ ]:
# Calculate daily returns for each stock
returns_df = merged_df.sort_values(
    ["nse_symbol", "trade_date"]
).copy()

returns_df["daily_return"] = (
    returns_df
    .groupby("nse_symbol")["close"]
    .pct_change()
)

print("=== DAILY RETURNS ===")
display(
    returns_df[
        ["nse_symbol", "company_name", "trade_date", "close", "daily_return"]
    ].head(15)
)

print("\n=== RETURN MISSING VALUES ===")
print(returns_df["daily_return"].isna().sum())

In [ ]:
# Calculate a 3-day rolling average closing price for each stock
returns_df = returns_df.sort_values(
    ["nse_symbol", "trade_date"]
).copy()

returns_df["rolling_3d_close"] = (
    returns_df
    .groupby("nse_symbol")["close"]
    .transform(lambda x: x.rolling(3).mean())
)

print("=== ROLLING 3-DAY CLOSE ===")
display(
    returns_df[
        ["nse_symbol", "trade_date", "close", "rolling_3d_close"]
    ].head(15)
)

In [ ]:
# Calculate 3-day rolling volatility for each stock
        returns_df["rolling_3d_volatility"] = (
    returns_df
    .groupby("nse_symbol")["daily_return"]
    .transform(lambda x: x.rolling(3).std())
)

print("=== 3-DAY ROLLING VOLATILITY ===")
display(
    returns_df[
        ["nse_symbol", "trade_date", "daily_return", "rolling_3d_volatility"]
    ].head(15)
)

## Phase 2.4 Completion

Core Pandas/data-handling operations have been demonstrated on the project's actual market and company-master datasets.

The current market snapshot contains only a small number of trading days, so return and rolling-volatility calculations here are demonstrations of the methodology rather than final investment features.

Meaningful historical features, targets, time-aware validation, and ML inputs are developed in later phases using a sufficiently long historical dataset.